# K513 · Week 5 Homework
## Classification — two models on the same question

This assignment covers **both** of this week's sessions: logistic regression on Tuesday and
decision trees on Thursday. You will build one of each on the same data, on the same split, and
then choose between them.

### Where your answers go

**Nothing you type in this notebook is collected.** The written answers are submitted through the
Canvas quiz **Week 5 Homework - Classification**, which also asks for a link to this notebook. Canvas records what you submit,
so editing the notebook afterwards cannot change what gets read — an answer left in here scores
zero.

The quiz is worth **100 points**. Most of them are for judgment, not for producing output.

---

### Before you type anything

**File → Save a copy in Drive.**

This notebook is read-only for you. You can type into it and run it and it will look completely
normal, but nothing you do will be saved. Save your own copy first, every time.

---

### Using AI in this notebook

Gemini is built into Colab and you are welcome to use it here. Two things worth knowing:

- It does not know which columns you have or what we covered in class. Whatever it writes, you own.
- The most useful thing you can ask it is **"explain what this line does"** — not "write it for me".

This week's trap: ask an AI to "find the best `max_depth`" and it will loop over depths, pick the one
with the highest **test** score, and hand it to you as the answer. That is the one move this course
has told you never to make. Nothing in your code says so, so the AI cannot know — the rule lives in
your head.

---

### Turn off Unwanted AI Assistance

AI-powered coding completion is turned on by default. It is convenient but does not give you a chance
to think and learn. Turning it off helps you learn. You can always turn it back on when needed.
- Tools → settings → AI Assistance → Uncheck "Show AI-powered inline code completions"
- Tools → settings → Uncheck "Show context-powered code completions"

---

### How to run a cell

Click on a cell, then press **Shift + Enter**. That runs it and moves you to the next one.

---
## The question

A maritime safety review is looking back at historical evacuation outcomes. They have one ship's
passenger list: 887 people, the features recorded about each of them before departure, and
whether they survived.

Two things they want to know:

1. **Was survival predictable from the recorded features at all?** — or was it, as is often
   assumed, close to chance.
2. **If it was, what was the pattern?** — in a form somebody could write into a procedure.

Those two questions want different models, which is the point of the assignment.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

pd.set_option('display.precision', 3)
np.set_printoptions(precision=3, suppress=True)

RANDOM_SEED = 42
TITANIC_URL = 'https://raw.githubusercontent.com/jl-uscn/k513-data/main/titanic.csv'

In [ ]:
titanic_df = pd.read_csv(TITANIC_URL)
print(titanic_df.shape)
titanic_df.head()

### The columns

| Column | What it is |
|---|---|
| `Survived` | whether the passenger survived — `Yes` or `No`. **This is the target.** |
| `Name` | the passenger's name |
| `Sex` | `male` or `female` |
| `Age` | age in years |
| `Siblings/Spouses Aboard` | how many were travelling with them |
| `Parents/Children Aboard` | how many were travelling with them |
| `Fare` | what they paid |
| `Pclass` | cabin class — `First`, `Second` or `Third` |

`Survived` is the **target**. Everything else was recorded before the ship sailed, so everything
else is a **candidate feature** — though one of them still should not go into the model, and Part 1
asks you which.

Look at it before you model it. The four questions of any variable still apply.

In [ ]:
titanic_df.info()

In [ ]:
titanic_df.describe()

In [ ]:
titanic_df['Survived'].value_counts(normalize=True)

---
## Part 1 · Set up the problem

**One column has to be left out of the model entirely.** Decide which, and be ready to say why.

In [ ]:
y = (titanic_df['Survived'] == ____).astype(int)
X = titanic_df.drop(columns=['Survived', ____])

categorical_features = ['Sex', ____]
continuous_features = ['Age', 'Siblings/Spouses Aboard',
                       'Parents/Children Aboard', 'Fare']

print(f"{len(X)} passengers, {X.shape[1]} columns to work with")
print(f"survived: {y.mean():.3f}")

**Split the data. Note that the test share is 30% this week, not the 25% we used in class.**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=____, random_state=RANDOM_SEED, stratify=____)

print(f"train {len(X_train)}   test {len(X_test)}")
print(f"survived — train {y_train.mean():.3f}   test {y_test.mean():.3f}")

**The baseline, before any model.** What share of the test passengers would you get right by
always guessing whichever outcome was commoner in the training data?

In [ ]:
baseline_test = ____
print(f"baseline on the test set: {baseline_test:.3f}")

> ✏️ **Answer in Canvas** — *Week 5 Homework - Classification*, question **(a)**.

**(a) · 10 points.** Which columns did you treat as **categorical** and which as
**continuous**, and which column did you leave out of the model altogether? Say why that column
had to go.

*Two or three sentences.*

---
## Part 2 · Model 1 — logistic regression

Tuesday's model, on new data. The numeric columns are standardized so the coefficients can be
compared with each other.

In [ ]:
logistic_pre = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), categorical_features),
    ('num', ____, continuous_features)])

logistic_model = Pipeline([('preprocessor', logistic_pre),
                           ('classifier', LogisticRegression(max_iter=1000,
                                                             random_state=RANDOM_SEED))])
logistic_model.fit(X_train, y_train)

print(f"train {logistic_model.score(X_train, y_train):.3f}   "
      f"test {logistic_model.score(X_test, y_test):.3f}")

### Which way does each column push?

In [ ]:
feature_names = (list(logistic_model.named_steps['preprocessor']
                     .named_transformers_['cat']
                     .get_feature_names_out(categorical_features))
                 + continuous_features)

coefficients = pd.Series(logistic_model.named_steps['classifier'].coef_[0],
                         index=feature_names).sort_values()

colors = ['#2E5EA8' if c > 0 else '#990000' for c in coefficients]
ax = coefficients.plot(kind='barh', color=colors)
ax.bar_label(ax.containers[0], fmt='%+.2f', padding=4)
ax.set_xlim(-3.2, 1.4)
ax.set_xlabel('pushes toward surviving  →')
plt.title('Which way does each column push?')
plt.show()

Every dropped level is a **reference level** — `OneHotEncoder(drop='first')` removed one
category from each column, and the remaining bars are read against it. Work out which two levels were
dropped before you write about any of these bars.

> ✏️ **Answer in Canvas** — *Week 5 Homework - Classification*, question **(b)**.

**(b) · 10 points.** Compare the logistic model to the baseline. State both numbers,
and say in plain English what the difference means for this question:

> *Was survival predictable from the recorded features at all — or was it, as is often assumed,
> close to chance?*

*Two or three sentences.*

---
## Part 3 · Model 2 — a decision tree

Thursday's model. Same split, same columns, one substitution and one removal.

In [ ]:
tree_pre = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), categorical_features),
    ('num', ____, continuous_features)])


def sweep_depths(depths):
    """Fit one tree per depth. Report both scores and how many rules it ended up with."""
    rows = []
    for d in depths:
        m = Pipeline([('preprocessor', tree_pre),
                      ('classifier', DecisionTreeClassifier(max_depth=d,
                                                            random_state=RANDOM_SEED))])
        m.fit(X_train, y_train)
        rows.append({'max_depth': d if d else 'none',
                     'leaves': m.named_steps['classifier'].get_n_leaves(),
                     'train': round(m.score(X_train, y_train), 3),
                     'test': round(m.score(X_test, y_test), 3)})
    out = pd.DataFrame(rows)
    out['gap'] = (out['train'] - out['test']).round(3)
    return out.set_index('max_depth')


sweep_depths([1, 2, 3, 4, 5, 6, 8, 10, None])

**Read that table before you choose a depth.** Two rows in it are worth stopping on: the
depth-1 tree, which asks exactly one question, and the unlimited tree at the bottom.

Now fit the depth you have decided to report, and draw it.

In [ ]:
final_tree = Pipeline([('preprocessor', tree_pre),
                       ('classifier', DecisionTreeClassifier(max_depth=____,
                                                             random_state=RANDOM_SEED))])
final_tree.fit(X_train, y_train)

tree_features = (list(final_tree.named_steps['preprocessor']
                      .named_transformers_['cat']
                      .get_feature_names_out(categorical_features))
                 + continuous_features)

plt.figure(figsize=(16, 8))
plot_tree(final_tree.named_steps['classifier'], feature_names=tree_features,
          class_names=['Died', 'Survived'], filled=True, rounded=True,
          impurity=False, label='root', precision=1, fontsize=10)
plt.show()

In [ ]:
print(export_text(final_tree.named_steps['classifier'],
                  feature_names=tree_features, show_weights=True))

> ✏️ **Answer in Canvas** — *Week 5 Homework - Classification*, question **(c)**.

**(c) · 15 points.** Write **two sentences a non-technical person would
understand**, both about the same data:

1. one **coefficient** from the logistic model — say which way it pushes and what it is being
   compared to
2. one **rule** from your tree — the path, what it concludes, and how many training passengers it
   was built on

Then say which of the two sentences you would rather put in a report, and why.

> ✏️ **Answer in Canvas** — *Week 5 Homework - Classification*, question **(d)**.

**(d) · 20 points.** Take **three rows** from your depth sweep: the depth you chose,
the depth with the highest test score, and the unlimited tree at the bottom.

For each one, name the **evaluation step** you land on and say what that step tells you to do.
Then say why you did **not** simply report the depth with the highest test score.

**Steps to Evaluate a Model** (Thursday of Week 4, PowerPoint page 11):

1. Score on train and on test. Always both, always in that order.
2. Both low? Too simple. Give it more to work with.
3. Train high, test far below? Too much model for the data you have. Get more data, or add a
   penalty.
4. Close together, and better than the baseline? Stop. This is what this data has.
5. Never choose a setting because it scored best on the test set.

---
## Part 4 · Choose between them

Both models are fitted on the same 620 passengers and scored on the same 267. Compare what they
**produce**, not just how they score.

In [ ]:
logistic_p = logistic_model.predict_proba(X_test)[:, 1]
tree_p = final_tree.predict_proba(X_test)[:, 1]

print(f"logistic: {len(np.unique(logistic_p.round(6)))} different scores")
print(f"tree:     {len(np.unique(tree_p.round(6)))} different scores")

Suppose the review wants a **shortlist**: the fifty passengers the model considers most
likely to have survived, so a historian can check them by hand. Each model has to produce fifty
names.

In [ ]:
def top_n(scores, n):
    """Take the n highest-scoring passengers and count how many really survived."""
    ranked = pd.DataFrame({'score': scores, 'really survived': y_test.values})
    ranked = ranked.sort_values('score', ascending=False)
    cut = ranked['score'].iloc[n - 1]
    return {'listed': n,
            'really survived': int(ranked.head(n)['really survived'].sum()),
            'score at the cut': round(cut, 3),
            'passengers tied there': int((ranked['score'] == cut).sum())}


pd.DataFrame([{'model': 'logistic', **top_n(logistic_p, 50)},
              {'model': 'tree', **top_n(____, 50)}]).set_index('model')

> ✏️ **Answer in Canvas** — *Week 5 Homework - Classification*, question **(e)**.

**(e) · 20 points.** The review has two questions:

> *1. Was survival predictable from the recorded features at all?*
> *2. If it was, what was the pattern — in a form somebody could write into a procedure?*

Say which model you would hand them for **each** question, and name a number from your own tables
above that supports each choice. If you would hand them the same model twice, say what that costs.

*Three or four sentences.*

---
## Part 5 · The Analyst's Note

> ✏️ **Answer in Canvas** — *Week 5 Homework - Classification*, question **Analyst's Note**.

**About 150 words · 20 points.**

The chair of the review has read your tables and asks:

> *"So what was recorded about each passenger predicts survival at about eighty percent. Does that
> mean the evacuation followed a rule? Should we write this pattern into our current guidance?"*

Write her a note that answers it.

A good note:

- gives her **a decision**, not a summary of what you tried
- quotes **numbers from your own tables**
- separates what the model **found** from what it **cannot establish** — this is one dataset, from
  one night, and a pattern in it is not a policy that was followed
- names the limitation you would want on the record if the recommendation went wrong
- does not use the words *accuracy*, *overfitting*, *max_depth* or *coefficient*

That last constraint is not a style exercise. If you can only explain the finding in the vocabulary
of the course, you cannot yet explain it to the person who has to act on it.

---
## Before you submit

1. **Run everything from the top.** *Runtime → Restart and run all.* Every cell should run with no
   errors and no blanks left in it.
2. **Share the notebook.** *Share → General access → **Anyone with the link*** → *Copy link*. Open
   the link in a private window and check that it loads.
3. **Open the Canvas quiz Week 5 Homework - Classification** and answer all seven questions there — the notebook link, (a) to
   (e), and the Analyst's Note.

**Nothing written in this notebook is read or graded.** The quiz is what gets marked.

| | points |
|---|---|
| Notebook link | 5 |
| (a) columns, and the one you left out | 10 |
| (b) against the baseline | 10 |
| (c) two sentences, and which you would report | 15 |
| (d) three depths against the evaluation steps | 20 |
| (e) which model for which question | 20 |
| Analyst's Note | 20 |
| **Total** | **100** |